# 03 Structure detection

A model in a file says nothing about blocks. This notebook takes a real energy system model as an
`.mps` file, finds a block structure for it, judges whether the structure is good, and only then
solves it. It is also the notebook to use for a model from any other tool.

[pipstools](https://pypi.org/project/pipstools/) offers two methods:

- regex: group the columns by a part of their names, for example the hour. Needs readable names.
- hypergraph: partition the matrix with mt-KaHyPar, without looking at names.

| sections | run on |
|---|---|
| 1 to 5: get the model, read its names, the regex method, judge the structure, compare block counts | every machine |
| 6: the hypergraph method | Linux, macOS and WSL (mt-KaHyPar has no Windows build) |
| 7: solve on one process | Linux or WSL with the `solver` extra |
| 8: solve on several processes with MPI | Linux or WSL with the `solver` extra |

In [ ]:
import workshop

workshop.check()

## 1. Get a model

The sample is a one-year, hourly [REMix](https://gitlab.com/dlr-ve/esy/remix/framework) model of
two regions with power plants, a battery storage and a transmission link: about 290,000 rows and
273,000 columns. It comes from the samples of the
[pipstools repository](https://gitlab.com/pips-ipmpp/pipstools) and is about 5 MB.

In [ ]:
import gzip
import shutil
import urllib.request
from pathlib import Path

URL = ("https://gitlab.com/pips-ipmpp/pipstools/-/raw/main/samples/remix.mps.gz"
       "?ref_type=heads&inline=false")
MODEL = Path("data") / "remix.mps"

MODEL.parent.mkdir(exist_ok=True)
if not MODEL.exists():
    print(f"downloading {URL}")
    urllib.request.urlretrieve(URL, MODEL)

with gzip.open(MODEL.with_suffix(".mps.gz"), 'rb') as f_in:
    with open(MODEL, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"{MODEL}  ({MODEL.stat().st_size / 1e6:.1f} MB)")



In [ ]:
# Brought your own model, or want to use one exported in 01? Point at it instead, and skip the
# cell above. .mps, .mps.gz, .lp and .lp.gz work, and so does a GAMS .gdx.
#
# Ideally the model should be below 1 million variables and constraints, for more you most likely 
# need more available memory and the structure detection will take longer. 
# 
# MODEL = Path("data/my_model.mps.gz")

In [ ]:
import highspy
import polars as pl

h = highspy.Highs()
h.silent()
h.readModel(str(MODEL)) # Make sure the model is unzipped, otherwise HiGHS might read an empty model
lp = h.getLp()
cols = pl.Series("name", lp.col_names_)
rows = pl.Series("name", lp.row_names_)
del h, lp

print(f"{len(cols):,} columns, {len(rows):,} rows\n")
print("some columns:", *cols.gather([0, 30, 50_000, 200_000]).to_list(), sep="\n  ")
print("some rows:   ", *rows.gather([0, 100_000, 250_000]).to_list(), sep="\n  ")

REMix names are readable: the symbol, then its indices in brackets. Most of them start with a time
step, `tm1` to `tm8760`, followed by region, year and technology. A few have no time step at all,
such as `converter_unitsBuild(CH_IT,2020,CCGT_CH4,2020)`: the capacity built for a technology,
which every hour of the year uses. That is exactly the root of the sample model in 00.

So the rule is: one block per slice of the year, taken from `tm...`.

## 3. Find a structure: regex

The pattern `(tm[0-9]+),` finds `tm` followed by digits and a comma, and the brackets capture
`tm399`. Every distinct captured value is a group; the groups are put into blocks in order.
Columns without a match go to the root. Test a pattern on the names before you hand it over:

In [ ]:
PATTERN = r"(tm[0-9]+),"

captured = cols.str.extract(PATTERN, 1)
print(f"{captured.is_not_null().sum():,} columns match, with {captured.n_unique() - 1:,} "
      f"distinct values")
print(f"{captured.is_null().sum():,} columns do not match and go to the root:")
print(cols.filter(captured.is_null()).str.extract(r"^([^(]+)", 1).value_counts(sort=True))

8,760 hours, and 26 columns without a time step: the capacities built (`...Build`) and in total
(`...Total`) per technology, storage and link. A good start for a root.

Now let pipstools do it, with 8 blocks of 1,095 hours each:

In [ ]:
BLOCKS = 8
workshop.pipstools("annotate", MODEL.name, "--regex-pattern", PATTERN, "--blocks", BLOCKS,
                   "--fileformat", "parquet", "--mpsreader", "highs",
                   "--outputfile", "remix_regex")
REGEX = f"data/remix_regex_{BLOCKS}b"

The lines to read, in order:

| line | what it tells you |
|---|---|
| `Annotating a model with ... rows, ... columns, ... nonzeros` | the size of the problem as read |
| `Partitioning of 8760 unique elements into 8 blocks` | what the pattern found |
| `Block 1: ...`, `Block 8: ...` | the first and last values that went into the first and last block |
| `Detected ... linking variables` | width of the left border, the root columns |
| `Detected ... linking constraints (thereof ... 2-link)` | height of the bottom border. A 2-link row touches only two neighbouring blocks and is much cheaper for PIPS-IPM++ than a global one |
| `Scoring components` | whitescore (the share of the matrix outside the borders, closer to 1 is better), the counts of global and 2-link rows, and an estimate of the size of the root system |
| `Total score` | the overall verdict of pipstools, higher is better, 100 at most |


<details><summary>💡 Hint: my model has years as well as hours</summary>

A pattern can capture more than one group, for example `(tm[0-9]+),.*,([0-9]{4}),` for the hour
and the year. `--regex-order` says which group sorts first: `--regex-order=[2,1]` orders by year,
then by hour, so blocks never mix two years. This sample has a single year, 2020, so it gives the
same structure; the groups show up as `2020_tm1`, `2020_tm2`, ...
</details>

## 4. Judge the structure

pipstools judges an annotated problem without solving it. `score` repeats the components from
above, as a standalone check you can run on any annotated problem, including one someone sent you.
`validate` checks that every nonzero sits where the annotation allows: inside its own leaf, in the
root, or in a linking row. Run it before shipping a problem to a cluster, so a job does not fail
an hour into a queue.

In [ ]:
workshop.pipstools("score", "remix_regex_8b.amat.parquet")
workshop.pipstools("validate", "remix_regex_8b.amat.parquet")

And the picture, with the numbers from the files:

In [ ]:
workshop.visualize(REGEX)

Eight leaf blocks along the diagonal, a left border of 26 columns and a bottom border of 32 rows,
too thin to see at this scale. That is the shape from 00, on a model with a quarter of a million
columns.

### What is in the borders?

The annotated problem is a set of parquet files with a simple schema, so you can look inside with
polars:

| file | columns |
|---|---|
| `<stem>.amat.parquet` | `row`, `col`, `value`, the matrix |
| `<stem>.cols.parquet` | `name`, `lower`, `upper`, `cost`, `partition` |
| `<stem>.rows.parquet` | `name`, `lower`, `upper`, `partition` |
| `<stem>.scalar.parquet` | objective bookkeeping |

The `partition` convention: columns `1` = root and `2 ... n+1` = the leaves; rows `1` = root rows,
`2 ... n+1` = local to a leaf, and `n+2` = linking.

In [ ]:
def border(stem):
    cols = pl.read_parquet(f"{stem}.cols.parquet")
    rows = pl.read_parquet(f"{stem}.rows.parquet")
    linking = int(cols["partition"].max()) + 1
    symbol = pl.col("name").str.extract(r"^([^(]+)", 1)
    root = cols.filter(pl.col("partition") == 1).group_by(symbol).len().sort("len")
    link = rows.filter(pl.col("partition") == linking).group_by(symbol).len().sort("len")
    return root, link, rows.filter(pl.col("partition") == linking)["name"]


root, link, linking_rows = border(REGEX)
print("root columns by symbol:", root, sep="\n")
print("linking rows by symbol:", link, sep="\n")
print("linking rows at tm1:", *linking_rows.filter(linking_rows.str.contains(r"\(tm1,")), sep="\n  ")

The root holds the 26 capacities, as the pattern test predicted. The linking rows are the storage
balance and losses of the battery in both regions, at the block boundaries: 2 regions x 7
boundaries = 14 of each kind, all 2-link.

And at `tm1`. REMix models the storage as cyclic: the first hour of the year follows the last. So
the rows at `tm1` tie block 1 to block 8, which are not neighbours. These are the 4 linking rows
that are not 2-link, the same effect as the cyclic storage exercise in the 01 notebooks.

<details><summary>💡 Hint: what is a "good" structure, in numbers?</summary>

There is no universal threshold. As a rule of thumb for energy models: a root of tens or hundreds
of columns rather than thousands, linking rows mostly 2-link, and leaves of about the same size
(the slowest leaf sets the pace). The regex structure here is about as good as it gets.
</details>

## 5. How many blocks?

Annotating is cheap, so try a few block counts. `workshop.structure` gives the numbers without the
picture:

In [ ]:
import re
import time

print(f"{'blocks':>6} {'root cols':>10} {'linking':>8} {'2-link':>7} {'score':>9} {'seconds':>8}")
for blocks in (2, 4, 8, 16, 32):
    t0 = time.perf_counter()
    out = workshop.pipstools("annotate", MODEL.name, "--regex-pattern", PATTERN,
                             "--blocks", blocks, "--fileformat", "parquet",
                             "--mpsreader", "highs", "--outputfile", "remix_regex",
                             quiet=True).stdout
    seconds = time.perf_counter() - t0
    info = workshop.structure(f"data/remix_regex_{blocks}b", quiet=True)
    score = re.search(r"Total score: ([\d.]+)", out)[1]
    print(f"{blocks:>6} {info['root_columns']:>10} {info['linking_rows']:>8} "
          f"{info['two_link_rows']:>7} {score:>9} {seconds:>8.1f}")

The root stays at 26 columns whatever the block count: the capacities do not depend on time.
Every extra block adds one boundary, and with it 4 more 2-link rows (balance and losses, two
regions). The 4 cyclic rows at `tm1` are the only ones that are not 2-link, except with 2 blocks:
then the first and the last block are neighbours, and every linking row is 2-link.

More blocks mean more parallelism and slightly taller borders. Choose at least as many blocks as
the processes you intend to run on, and enough hours per block that each leaf is worth a process.

## 6. Find a structure: hypergraph

When the names carry no structure, for example in benchmark files whose names were stripped to
`x0, x1, ...`, a partitioner has to find it in the matrix. pipstools builds a hypergraph from the
matrix and lets mt-KaHyPar cut it into blocks with as few cut edges as possible. The same REMix
model, without using its names:

In [ ]:
%%requires mtkahypar
workshop.pipstools("annotate", MODEL.name, "--blocks", BLOCKS, "--fileformat", "parquet",
                   "--mpsreader", "highs", "--outputfile", "remix_hg")
workshop.visualize(f"data/remix_hg_{BLOCKS}b")

A total score of 0, although the picture looks almost as clean as the regex one. The difference
is in the top left corner: the partitioner put about 7,000 columns and rows into the root. The
`est_root_dim` and `est_root_nnz` components of the score estimate the system PIPS-IPM++ would have
to solve for the root: over 14,000 rows with about 100 million nonzeros, dense and held by every
process. That would dominate the whole solve.

Two new lines appear in the output:

| line | what it tells you |
|---|---|
| `Detected ... dense columns` | columns touching very many rows. They go to the root: in a leaf they would make every row they touch linking |
| `Detected ... rows with a single column` | plain variable bounds; not really constraints, set aside |

The hypergraph can be built in different ways. By default the columns are the nodes (`col`);
`--hypergraph row` makes the rows the nodes instead:

In [ ]:
%%requires mtkahypar
workshop.pipstools("annotate", MODEL.name, "--blocks", BLOCKS, "--hypergraph", "row",
                   "--fileformat", "parquet", "--mpsreader", "highs",
                   "--outputfile", "remix_hgrow")
workshop.visualize(f"data/remix_hgrow_{BLOCKS}b")

29 root columns and no linking rows at all: nearly as good as the regex, found without any names.
The partitioner resolved every coupling by moving a column into the root instead of leaving a row
linking.

The lesson: check a structure, however you got it. A partitioner optimises its own objective, which
is not automatically yours, and its settings matter. When the names carry the structure, the regex
method is free, exact and reproducible; use it first.

Things worth trying one at a time: `--hypergraph colrow`, `--hg-objective km1` or `soed` (what the
partitioner minimises), and other block counts.

<details><summary>💡 Hint: does the hypergraph result change between runs?</summary>

Partitioning is a heuristic. pipstools seeds it, so the same input and version give the same
result, but a different version or block count can land somewhere slightly different. Judge a
structure by its borders and score, not by which exact rows ended up where.
</details>

## 7. Solve it

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped; section 9 shows how to take the files to a machine that can solve them.

The annotated files are all PIPS-IPM++ needs. `read_monolithic` loads them as a
`StructuredProblem`, and `solve` solves it in this notebook, on one process. This is a real model,
so expect a solve of a few minutes and about 2 GB of memory.

In [ ]:
%%requires solver
import pipsipmpp

problem = pipsipmpp.read_monolithic(REGEX)
print(f"{problem.n_vars:,} variables in {problem.n_blocks} leaves, "
      f"{int((problem.var_block == 0).sum())} in the root")

t0 = time.perf_counter()
result = pipsipmpp.solve(problem, options={"SILENT": True})
seconds = time.perf_counter() - t0

In [ ]:
%%requires solver
print(f"{result.status.name}  objective {result.objective:.6e}  "
      f"{result.iterations} iterations  {seconds:.0f} s")

<details><summary>💡 Hint: what is that objective?</summary>

The total system cost of the modelled year, in the currency unit of the model.
</details>

<details><summary>💡 Hint: the solve died, or the kernel restarted</summary>

The operating system stopped it for lack of memory. Close other programs and restart the kernel,
then run sections 1, 3 and 7 only. The notebook holds the names from section 2 in memory as well.
</details>

## 8. Solve on several processes

A notebook is one process. For several, the same solve runs as a script under MPI. Only process 0
reads the files; the others receive their blocks from it.

In [ ]:
%%writefile solve_file.py
import sys
import time

import pipsipmpp
from mpi4py import MPI

stem = sys.argv[1]
comm = MPI.COMM_WORLD
t0 = time.perf_counter()
problem = pipsipmpp.read_monolithic(stem) if comm.Get_rank() == 0 else None
result = pipsipmpp.solve(problem, comm, options={"SILENT": True})
if comm.Get_rank() == 0:
    print(f"ranks={comm.Get_size()} {result.status.name} objective={result.objective:.6e} "
          f"iterations={result.iterations} wall={time.perf_counter() - t0:.0f}s")

In [ ]:
%%requires solver
import os

ranks = min(BLOCKS, os.cpu_count() or 1)      # at most one process per leaf block
workshop.mpirun("solve_file.py", ranks, REGEX, show="ranks=")

The same objective as on one process. Whether it is faster depends on your machine: the leaves are
worked on at the same time, but a laptop has few cores and shares its memory bandwidth between
them. On a cluster node the picture changes.

<details><summary>💡 Hint: how many processes should I use?</summary>

At most one per leaf block (PIPS-IPM++ refuses more), and on a laptop at most the number of CPU
cores. Fewer processes than leaves is fine; leaves then share a process. The cell takes the
smaller of the two.
</details>

## 9. Hand it over

The annotated problem is self-contained: the machine that solves it needs neither your modelling
tool nor the original `.mps`. From macOS or Windows, copy these files to WSL or a Linux cluster:

In [ ]:
total = 0
for f in sorted(Path(REGEX).parent.glob(f"{Path(REGEX).name}.*.parquet")):
    total += f.stat().st_size
    print(f"{f.stat().st_size / 1e6:8.2f} MB  {f.name}")
print(f"{total / 1e6:8.2f} MB  total")

and there, in the `notebooks` folder of the workshop after `uv sync --extra solver` or
`pixi install -e solver`, with `solve_file.py` from section 8:

```console
$ uv run --extra solver mpiexec -n 8 python solve_file.py data/remix_regex_8b
$ pixi run -e solver mpiexec -n 8 python solve_file.py data/remix_regex_8b
```

The point of annotating locally is that finding a good decomposition takes several tries, and you
do not want to queue a cluster job for each one.

## 10. Now your own model

Point `MODEL` at your own file (section 1) and ask:

1. Do the names carry an index? Then start with regex: free, exact, reproducible. Test the
   pattern on the names first, as in section 3.
2. How wide are the borders? Tens or hundreds of root columns and mostly 2-link rows mean it will
   pay off; thousands of global linking rows mean it will not.
3. Does the picture look block-diagonal? A scatter means the model is not structured along the
   index you split; try another one.
4. What happens with 4, 8, 16 blocks?

If the borders stay large whatever you try, that is a real answer too: the model may be coupled
by something that could be reformulated. Bring it to the wrap-up.

## 11. Checkpoint

- Why do the capacities end up in the root, both with the regex and with the row hypergraph?
- Why does the cyclic storage of REMix create rows that are not 2-link?
- The default hypergraph found a structure with a good whitescore and a total score of 0. Why?

## Next

[04 Deep dive](04-deep-dive.ipynb): building the matrices yourself, the distributed layout in
which no process ever holds the whole model, and reading a solution back.